In [2]:
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path
from rasterstats import zonal_stats

# =========================================================
# PATHS
# =========================================================

ROOT = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land"
)

heatday_dir = ROOT / "data" / "monthly_heatday_scores"
output_dir = ROOT / "data" / "variables"

geojson_path = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/Maps/od_ids-drr_shapefiles/odisha_block_final_reduced.geojson"
)

output_dir.mkdir(parents=True, exist_ok=True)

# =========================================================
# LOAD GEOJSON
# =========================================================

gdf = gpd.read_file(geojson_path)

if "object_id" not in gdf.columns:
    raise ValueError("object_id column missing")

gdf = gdf[["object_id", "geometry"]]

# =========================================================
# MASTER STORAGE
# =========================================================

all_records = []

# =========================================================
# LOOP
# =========================================================

years = [2023, 2024]

for year in years:

    print(f"\n================ YEAR {year} ================")

    for month in range(1, 13):

        timeperiod = f"{year}_{month:02d}"

        raster_path = heatday_dir / f"HEATDAY_{year}_{month:02d}.tif"

        print(f"\nProcessing {timeperiod}")

        if not raster_path.exists():
            print("SKIP missing raster")
            continue

        # =================================================
        # ZONAL STATS
        # =================================================

        zs = zonal_stats(
            vectors=gdf,
            raster=str(raster_path),
            stats=["mean"],
            geojson_out=True,
            nodata=np.nan
        )

        # =================================================
        # STORE RESULTS
        # =================================================

        for f in zs:

            record = {
                "object_id": f["properties"]["object_id"],
                "mean-heatday": f["properties"]["mean"],
                "timeperiod": timeperiod
            }

            all_records.append(record)

        # =================================================
        # ALSO WRITE MONTHLY FILE
        # =================================================

        df_month = pd.DataFrame([
            {
                "object_id": f["properties"]["object_id"],
                "mean-heatday": f["properties"]["mean"],
                "timeperiod": timeperiod
            }
            for f in zs
        ])

        out_csv = output_dir / f"HEATDAY_{year}_{month:02d}.csv"
        df_month.to_csv(out_csv, index=False)

        print("Saved:", out_csv)

# =========================================================
# FINAL MASTER CSV
# =========================================================

df_all = pd.DataFrame(all_records)

final_csv = output_dir / "heatdays.csv"
df_all.to_csv(final_csv, index=False)

print("\n================ DONE ================")
print("MASTER FILE SAVED:", final_csv)


================ YEAR 2023 ================

Processing 2023_01
Saved: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_01.csv

Processing 2023_02
Saved: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_02.csv

Processing 2023_03
Saved: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_03.csv

Processing 2023_04
Saved: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_04.csv

Processing 2023_05
Saved: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_05.csv

Processing 2023_06
Saved: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land/data/variables/HEATDAY_2023_06.csv

Processing 2023_07
Saved: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/

In [ ]:
# the below block is for testing code

In [6]:
import numpy as np
import rasterio
import calendar
from pathlib import Path

# =========================================================
# CONFIGURE — only change these two lines
# =========================================================
YEAR  = 2023
MONTH = 3    # ← change this (1–12)

# =========================================================
# PATHS
# =========================================================
quarter_dir   = Path("/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land/data/quarterly_tiffs")
threshold_dir = Path("/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land/data/percentile_tiffs")

if MONTH in [1,2,3]:   season, q = "JFM", "Q1"
elif MONTH in [4,5,6]: season, q = "AMJ", "Q2"
elif MONTH in [7,8,9]: season, q = "JAS", "Q3"
else:                  season, q = "OND", "Q4"

quarter_first_month = ((MONTH - 1) // 3) * 3 + 1
day_offset    = sum(calendar.monthrange(YEAR, m)[1] for m in range(quarter_first_month, MONTH))
days_in_month = calendar.monthrange(YEAR, MONTH)[1]
band_start    = day_offset + 1
band_end      = day_offset + days_in_month

hi_path = quarter_dir / f"HI_{YEAR}_{q}.tif"

print(f"\n{'='*55}")
print(f"  {YEAR}-{MONTH:02d}  |  Quarter={q}  Season={season}")
print(f"  Bands {band_start}–{band_end}  ({days_in_month} days)")
print(f"{'='*55}")

# =========================================================
# LOAD HI SLICE
# =========================================================
with rasterio.open(hi_path) as src:
    total_bands = src.count
    if band_end > total_bands:
        print(f"!! BAND ERROR: need band {band_end} but TIF only has {total_bands}")
    hi = src.read(list(range(band_start, band_end + 1)))

print(f"\nHI values  min={np.nanmin(hi):.2f}  max={np.nanmax(hi):.2f}  mean={np.nanmean(hi):.2f}")
if np.nanmean(hi) > 200:
    print("!! UNIT WARNING: values look like Kelvin — check unit consistency with thresholds")

# =========================================================
# LOAD THRESHOLDS
# =========================================================
def load(path):
    with rasterio.open(path) as src:
        return src.read(1)

p80 = load(threshold_dir / f"{season}_P80_1990_2023.tif")
p88 = load(threshold_dir / f"{season}_P88_1990_2023.tif")
p95 = load(threshold_dir / f"{season}_P95_1990_2023.tif")
p99 = load(threshold_dir / f"{season}_P99_1990_2023.tif")

print(f"\nThresholds (spatial mean across grid):")
print(f"  P80={np.nanmean(p80):.2f}  P88={np.nanmean(p88):.2f}  "
      f"P95={np.nanmean(p95):.2f}  P99={np.nanmean(p99):.2f}")

# =========================================================
# SCORE COMPUTATION
# =========================================================
monthly = np.zeros(hi.shape[1:], dtype=np.float32)
for d in range(hi.shape[0]):
    h = hi[d]
    s = np.zeros_like(h, dtype=np.uint8)
    s[(h >= p80) & (h < p88)] = 1
    s[(h >= p88) & (h < p95)] = 2
    s[(h >= p95) & (h < p99)] = 3
    s[h >= p99]               = 4
    monthly += s

print(f"\nMonthly score  min={np.nanmin(monthly):.1f}  "
      f"max={np.nanmax(monthly):.1f}  mean={np.nanmean(monthly):.2f}")
print(f"  Pixels score>0 : {int(np.sum(monthly > 0))}")
print(f"  Pixels score=0 : {int(np.sum(monthly == 0))}")

# =========================================================
# CENTRE PIXEL TRACE
# =========================================================
r, c = hi.shape[1] // 2, hi.shape[2] // 2
print(f"\nDay-by-day trace — centre pixel (row={r}, col={c}):")
print(f"  Thresholds here: P80={p80[r,c]:.2f}  P88={p88[r,c]:.2f}  "
      f"P95={p95[r,c]:.2f}  P99={p99[r,c]:.2f}")
print(f"  {'Day':<5} {'HI':>8}  {'Score':>5}  {'Band':>5}")
for d in range(hi.shape[0]):
    h = hi[d, r, c]
    s = 4 if h>=p99[r,c] else 3 if h>=p95[r,c] else 2 if h>=p88[r,c] else 1 if h>=p80[r,c] else 0
    flag = " ←" if s > 0 else ""
    print(f"  {d+1:<5} {h:>8.3f}  {s:>5}  {band_start+d:>5}{flag}")

pixel_total = int(monthly[r, c])
print(f"\n  Centre pixel monthly total : {pixel_total}")
if pixel_total == 0 and np.nanmax(hi[:, r, c]) < p80[r, c]:
    print(f"  → HI never reaches P80 ({p80[r,c]:.2f}) — score=0 is physically correct")


  2023-03  |  Quarter=Q1  Season=JFM
  Bands 60–90  (31 days)

HI values  min=21.57  max=40.16  mean=31.93

Thresholds (spatial mean across grid):
  P80=32.94  P88=34.23  P95=35.73  P99=37.43

Monthly score  min=0.0  max=25.0  mean=4.95
  Pixels score>0 : 2157
  Pixels score=0 : 2463

Day-by-day trace — centre pixel (row=30, col=38):
  Thresholds here: P80=31.19  P88=32.45  P95=33.95  P99=35.57
  Day         HI  Score   Band
  1       29.960      0     60
  2       30.258      0     61
  3       30.319      0     62
  4       30.310      0     63
  5       30.742      0     64
  6       30.326      0     65
  7       30.499      0     66
  8       30.005      0     67
  9       29.079      0     68
  10      29.314      0     69
  11      29.593      0     70
  12      29.975      0     71
  13      30.365      0     72
  14      30.128      0     73
  15      30.612      0     74
  16      29.902      0     75
  17      27.592      0     76
  18      28.107      0     77
  19      25

In [1]:
import json
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from pathlib import Path

# =========================================================
# PATHS
# =========================================================

ROOT = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land"
)

heat_dir = ROOT / "data" / "summer_heatday_scores"

geojson_path = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/assets/district.geojson"
)

output_dir = ROOT / "data" / "district_heat_risk"
output_dir.mkdir(parents=True, exist_ok=True)

# =========================================================
# LOAD GEOJSON
# =========================================================

gdf = gpd.read_file(geojson_path)

# FORCE correct field (your fix)
if "dtname" not in gdf.columns:
    raise ValueError("GeoJSON does NOT contain 'dtname' column")

district_names = gdf["dtname"].astype(str).tolist()

# geometries
geoms = json.loads(gdf.to_json())["features"]
geoms = [f["geometry"] for f in geoms]

# sanity check
assert len(district_names) == len(geoms), "Mismatch in dtname and geometry count"

# =========================================================
# YEARS
# =========================================================

years = [2021, 2022, 2023, 2024, 2025]

# =========================================================
# ZONAL MEAN FUNCTION
# =========================================================

def zonal_mean(raster_src, geometries):

    means = []

    for geom in geometries:

        out, _ = mask(
            raster_src,
            [geom],
            crop=True,
            filled=True,
            nodata=np.nan
        )

        arr = out[0]
        means.append(np.nanmean(arr))

    return np.array(means)

# =========================================================
# COLLECT DATA
# =========================================================

all_years = []

for year in years:

    path = heat_dir / f"SUMMER_HEATDAY_{year}.tif"

    if not path.exists():
        print(f"Missing {path}")
        continue

    print(f"\nProcessing {year}")

    with rasterio.open(path) as src:
        means = zonal_mean(src, geoms)

    df_year = pd.DataFrame({
        "district": district_names,
        "year": year,
        "summer_heatdays": means
    })

    all_years.append(df_year)

# =========================================================
# MERGE
# =========================================================

df = pd.concat(all_years, ignore_index=True)

# =========================================================
# 5-YEAR MEAN PER DISTRICT
# =========================================================

district_mean = (
    df.groupby("district", as_index=False)
      .agg(mean_heatdays=("summer_heatdays","sum"))
)

# =========================================================
# Z-SCORE
# =========================================================

mu = district_mean["mean-heatdays"].mean()
sigma = district_mean["mean-heatdays"].std(ddof=0)

district_mean["zscore"] = (
    district_mean["mean-heatdays"] - mu
) / sigma

# =========================================================
# CLASSIFICATION (1–5)
# =========================================================

# def classify(z):

#     if z <= -1.5:
#         return 1
#     elif z <= -0.5:
#         return 2
#     elif z <= 0.5:
#         return 3
#     elif z <= 1.5:
#         return 4
#     else:
#         return 5

# district_mean["heat_risk_class"] = district_mean["zscore"].apply(classify)


import jenkspy
import numpy as np

# Number of classes
n_classes = 5

# Compute Jenks breaks
breaks = jenkspy.jenks_breaks(
    district_mean["zscore"].dropna().values,
    n_classes=n_classes
)

print("Jenks breaks:", breaks)

# Classify into 1–5 using the Jenks breaks
district_mean["heat_risk_class"] = np.digitize(
    district_mean["zscore"],
    breaks[1:-1],   # exclude min and max
    right=True
) + 1

# Preserve NaNs if any
district_mean.loc[district_mean["zscore"].isna(), "heat_risk_class"] = np.nan
district_mean["heat_risk_class"] = district_mean["heat_risk_class"].astype("Int64")

# =========================================================
# SAVE OUTPUT
# =========================================================

out_file = output_dir / "district_summer_heatrisk_2021_2025.csv"

district_mean.to_csv(out_file, index=False)

print("\n===================================")
print("SAVED:", out_file)
print("===================================")


Processing 2021

Processing 2022

Processing 2023

Processing 2024

Processing 2025


KeyError: 'mean-heatdays'